# 03 · Context Window

**Idea:** a model can only attend to a fixed number of tokens. Go over the limit and older content must be dropped or summarized. Here we simulate a growing conversation and apply a simple **truncation** strategy.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'study_buddy').exists())
sys.path.insert(0, str(root))

from study_buddy.tokens import count_message_tokens

In [ ]:
# Pretend the model window is tiny so we can hit the limit quickly.
WINDOW = 120  # tokens

history = [{'role': 'system', 'content': 'You are a helpful assistant.'}]
for i in range(1, 11):
    history.append({'role': 'user', 'content': f'Message number {i}. ' * 4})
    history.append({'role': 'assistant', 'content': f'Reply to message {i}. ' * 4})

print('full history tokens:', count_message_tokens(history))

In [ ]:
def truncate_to_window(messages, window):
    """Keep the system message + most recent turns that fit the window."""
    system = [m for m in messages if m['role'] == 'system']
    rest = [m for m in messages if m['role'] != 'system']
    kept = []
    for m in reversed(rest):
        trial = system + [m] + kept
        if count_message_tokens(trial) > window:
            break
        kept.insert(0, m)
    return system + kept

pruned = truncate_to_window(history, WINDOW)
print('kept messages:', len(pruned), 'of', len(history))
print('pruned tokens:', count_message_tokens(pruned), '(<=', WINDOW, ')')
for m in pruned:
    print(' ', m['role'], '->', m['content'][:40])

**Takeaway:** managing the window (truncate, summarize, or retrieve) is a core agent skill. RAG (topic 15) is one answer to "my knowledge doesn't fit".

Next → `04_embeddings.ipynb`